In [1]:
import pandas as pd
import numpy as np 
from scipy import stats as stat
from scipy.stats import bootstrap as bootstrap
import matplotlib.pyplot as plt
import seaborn as sns
from seaborn import catplot
import random as rd
import os
from sklearn.preprocessing import StandardScaler as ss
from sklearn.decomposition import PCA as pca
import math
from scipy.stats import linregress
import mne
%matplotlib qt
import os
from itertools import compress

import mne
from mne import datasets
import numpy as np
from nilearn.plotting import plot_design_matrix

import mne_nirs
import mne_bids
# Import StatsModels
import statsmodels.formula.api as smf
import matplotlib as mpl

# Import MNE processing
from mne.preprocessing.nirs import beer_lambert_law, optical_density

# Import MNE-BIDS processing
from mne_bids import BIDSPath, get_entity_vals, read_raw_bids,stats

from mne_nirs.channels import get_long_channels, get_short_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.io.fold import fold_channel_specificity

# Import MNE-NIRS processing
from mne_nirs.statistics import run_glm, statsmodels_to_results
from mne_nirs.visualisation import plot_glm_group_topo, plot_glm_surface_projection 
import os.path as op

from mne.datasets.testing import data_path
from mne.viz import set_3d_view

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.ar_model import AutoReg
import statsmodels.api as sm
# Import common libraries
from collections import defaultdict
from copy import deepcopy
from itertools import compress
from pprint import pprint

# Import Plotting Library
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import StatsModels
import statsmodels.formula.api as smf
from mne import Epochs, events_from_annotations, set_log_level
from mne.preprocessing.nirs import (
    beer_lambert_law,
    optical_density,
    scalp_coupling_index,
    temporal_derivative_distribution_repair,
)

# Import MNE processing
from mne.viz import plot_compare_evokeds

# Import MNE-BIDS processing
from mne_bids import BIDSPath, read_raw_bids

# Import MNE-NIRS processing
from mne_nirs.channels import get_long_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.signal_enhancement import enhance_negative_correlation

C:\Users\pkell\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [ ]:
## Load in Functional Connectivity Data From MATLAB 
file = r"C:\Users\pkell\OneDrive\Documents\MATLAB\EI_ISC_group_functional_conn.csv"
df = pd.read_csv(file)

In [185]:
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\functional_conn_subjectlevel_pruned2hz_cond.csv"
df1 = pd.read_csv(file1)

In [19]:
# Filtering the data by condition (if you have different conditions as opposed to task vs rest connectivity)
df1 = df[df["condition"] == 1]
df2 = df[df["condition"] == 2]

## Correlate vector of t values from 1st order to T vector from second order

from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# # --- Ensure data is sorted consistently ---
# df1 = df_cond1.sort_values(by=['ROI_X', 'ROI_Y', 'chOrigin', 'chDest']).reset_index(drop=True)
# df2 = df_cond2.sort_values(by=['ROI_X', 'ROI_Y', 'chOrigin', 'chDest']).reset_index(drop=True)

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df1 = df1[~df1['DetectorOrigin'].isin(bad_detectors)]
df2 = df2[~df2['DetectorOrigin'].isin(bad_detectors)]
df1 = df1[~df1['DetectorDest'].isin(bad_detectors)]
df2 = df2[~df2['DetectorDest'].isin(bad_detectors)]

# # --- Drop Specific Source-Detector Pairs ---
# df1 = df1[~((df1['source'] == 14) & (df1['detector'] == 13))]
# df2 = df2[~((df2['source'] == 16) & (df2['detector'] == 12))]

# --- Remove mirror-flipped channel pairs ---
def make_pair_id(df):
    return df.apply(
        lambda row: tuple(sorted([
            (row['SourceOrigin'], row['DetectorOrigin']),
            (row['SourceDest'], row['DetectorDest'])
        ])), axis=1)

df1['pair_id'] = make_pair_id(df1)
df2['pair_id'] = make_pair_id(df2)

# Keep only one direction per pair
df1 = df1.drop_duplicates(subset='pair_id')
df2 = df2.drop_duplicates(subset='pair_id')

# Drop the helper column
df1 = df1.drop(columns='pair_id')
df2 = df2.drop(columns='pair_id')

# --- Create the r-value vectors ---
r_vec_cond1 = df1['R'].values
r_vec_cond2 = df2['R'].values

# --- Double-check that vectors have the same length ---
if len(r_vec_cond1) != len(r_vec_cond2):
    raise ValueError("Condition 1 and Condition 2 vectors do not have the same length! Check preprocessing.")
else:
    print(f"✅ Matched number of channel pairs: {len(r_vec_cond1)}")

# --- Remove pairs where either condition has NaN ---
valid_mask = np.isfinite(r_vec_cond1) & np.isfinite(r_vec_cond2)
r_vec_cond1_clean = r_vec_cond1[valid_mask]
r_vec_cond2_clean = r_vec_cond2[valid_mask]

print(f"✅ Number of valid (non-NaN) pairs: {len(r_vec_cond1_clean)}")

# --- Correlate the cleaned vectors ---
r_across_conditions, p_value_across_conditions = pearsonr(r_vec_cond1_clean, r_vec_cond2_clean)

# --- Print the results ---
print(f"📈 Pearson correlation between Condition 1 and Condition 2 channel-level r values (after removing NaNs):")
print(f"r = {r_across_conditions:.3f}, p = {p_value_across_conditions:.30f}")

# --- Optional: Visualize ---
plt.figure(figsize=(7, 6))
sns.regplot(x=r_vec_cond1_clean, y=r_vec_cond2_clean, scatter_kws={'s':30, 'alpha':0.7})

plt.title('Channel-level FC Correlation: Condition 1 vs Condition 2', fontsize=14)
plt.xlabel('Condition 1 r-values')
plt.ylabel('Condition 2 r-values')
plt.annotate(f"r = {r_across_conditions:.2f}\np = {p_value_across_conditions:.4f}",
             xy=(0.05, 0.95), xycoords='axes fraction',
             ha='left', va='top', fontsize=11,
             bbox=dict(boxstyle="round", facecolor='white', edgecolor='gray', alpha=0.8))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


✅ Matched number of channel pairs: 741
✅ Number of valid (non-NaN) pairs: 703
📈 Pearson correlation between Condition 1 and Condition 2 channel-level r values (after removing NaNs):
r = 0.997, p = 0.000000000000000000000000000000


In [ ]:
# df

In [2]:
# Define Regions of Interest (ROI)
roi_definitions = {
    "Left Frontal": [
        'S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo', 'S2_D3 hbo',
        'S3_D1 hbo', 'S3_D3 hbo', 'S4_D3 hbo', 'S4_D2 hbo', 'S4_D4 hbo',
        'S5_D3 hbo', 'S5_D4 hbo', 'S5_D16 hbo'
    ],
    "Left Parietal": [
        'S11_D9 hbo', 'S11_D8 hbo', 'S12_D8 hbo', 'S12_D10 hbo', 'S13_D8 hbo',
        'S13_D9 hbo', 'S13_D10 hbo', 'S13_D11 hbo', 'S12_D19 hbo', 'S11_D18 hbo', 'S13_D20 hbo'
    ],
    "Right Frontal": [
        'S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo', 'S7_D7 hbo',
        'S8_D5 hbo', 'S8_D7 hbo', 'S9_D7 hbo', 'S9_D6 hbo', 'S10_D7 hbo', 'S10_D17 hbo'
    ],
    "Right Parietal": [
        'S14_D12 hbo', 'S14_D13 hbo', 'S15_D12 hbo', 'S15_D14 hbo', 'S16_D12 hbo',
        'S16_D13 hbo', 'S16_D14 hbo', 'S16_D15 hbo', 'S14_D21 hbo', 'S15_D22 hbo', 'S16_D23 hbo'
    ]
}

In [7]:
# Define Regions of Interest (Channel)
roi_definitions = {
    "Left RLPFC": [
        'S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'
    ],
    "Left DLPFC": [
          'S4_D3 hbo', 'S4_D2 hbo', 'S4_D4 hbo','S5_D3 hbo', 'S5_D4 hbo', 'S5_D16 hbo','S2_D3 hbo'
    ],
    "Left VLPFC": [
         'S3_D1 hbo', 'S3_D3 hbo'
    ],
    "Left Sup Parietal": [
         'S12_D8 hbo', 'S12_D10 hbo', 'S13_D10 hbo','S13_D20 hbo', 'S12_D19 hbo'
    ],
    "Left Inf Parietal": [
        'S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo', 'S13_D9 hbo', 'S13_D11 hbo', 'S11_D18 hbo',
    ],
    "Right RLPFC": [
        'S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo', 
    ],
    "Right DLPFC": [
         'S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo', 'S10_D7 hbo', 'S10_D17 hbo'
    ],
    "Right VLPFC": [
         'S9_D7 hbo', 'S9_D6 hbo'
    ],
    "Right Sup Parietal": [
        'S14_D12 hbo', 'S14_D13 hbo', 
         'S14_D21 hbo', 'S16_D13 hbo'
    ],
    "Right Inf Parietal": [
         'S15_D12 hbo', 'S15_D14 hbo','S16_D12 hbo', 'S16_D14 hbo', 'S16_D15 hbo', 'S15_D22 hbo','S16_D23 hbo' 
    ]
}

In [ ]:
# Define Regions of Interest (ROI Aggreation)
roi_definitions = {
    "Left RLPFC": [
        'S2_D2 hbo'
    ],
    "Left DLPFC": [
          'S4_D3 hbo'
    ],
    "Left VLPFC": [
         'S3_D1 hbo'
    ],
    "Left Sup Parietal": [
          'S12_D10 hbo'
    ],
    "Left Inf Parietal": [
        'S11_D9 hbo'
    ],
    "Right RLPFC": [
        'S7_D5 hbo'
    ],
    "Right DLPFC": [
         'S8_D7 hbo'
    ],
    "Right VLPFC": [
         'S9_D6 hbo'
    ],
    "Right Sup Parietal": [
        'S14_D13 hbo' 
         
    ],
    "Right Inf Parietal": [
         'S15_D14 hbo' 
    ]
}

In [23]:
## Run this Formatting Code 

df_hbo = df[df['TypeOrigin'] == 'hbo']
df_hbo = df_hbo[df_hbo['TypeDest'] == 'hbo']

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df_hbo = df_hbo[~df_hbo['DetectorOrigin'].isin(bad_detectors)]
df_hbo = df_hbo[~df_hbo['DetectorDest'].isin(bad_detectors)]

# Create a new column 'chOrigin' with the structure "S<SourceOrigin>_D<DetectorOrigin> <TypeOrigin>"
df_hbo['chOrigin'] = (
    'S' + df_hbo['SourceOrigin'].astype(str) +
    '_D' + df_hbo['DetectorOrigin'].astype(str) +
    ' ' + df_hbo['TypeOrigin']
)

# Create a new column 'chDest' with the structure "S<SourceDest>_D<DetectorDest> <TypeDest>"
df_hbo['chDest'] = (
    'S' + df_hbo['SourceDest'].astype(str) +
    '_D' + df_hbo['DetectorDest'].astype(str) +
    ' ' + df_hbo['TypeDest']
)

# Optionally, if you no longer need the original columns, you can drop them:
df_hbo = df_hbo.drop(columns=[
    'SourceOrigin', 'DetectorOrigin', 'TypeOrigin',
    'SourceDest', 'DetectorDest', 'TypeDest'
])

# Inspect the modified dataframe
#df_hbo


In [25]:
## Uncomment (select) which ROI configuration your using (channel/ROI AGG) - change to match your montage 
## Run formatting Code

import pandas as pd

# ## Define your ROI definitions (as provided)
# # Define ROIs (HbO only) - EI data
# roi_definitions = {
#     "Left RLPFC": ['S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'],
#     "Left VLPFC": ['S3_D1 hbo', 'S3_D3 hbo'],
#     "Left DLPFC": ['S2_D3 hbo', 'S4_D3 hbo', 'S4_D2 hbo'],
#     "Left OFC": ['S1_D15 hbo'],
#     "Left STS": ['S5_D4 hbo','S5_D10 hbo','S11_D4 hbo', 'S11_D10 hbo', 'S12_D9 hbo', 'S12_D10 hbo'],
#     "Left TPJ": ['S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo','S13_D9 hbo'],
#     "Right RLPFC": ['S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo'],
#     "Right DLPFC": ['S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo'],
#     "Right VLPFC": ['S9_D7 hbo', 'S9_D6 hbo'],
#     "Right OFC": ['S6_D15 hbo'],
#     "Right STS": ['S10_D13 hbo','S10_D11 hbo','S14_D11 hbo', 'S14_D14 hbo','S15_D11 hbo','S15_D13 hbo'],
#     "Right TPJ": ['S15_D12 hbo', 'S15_D14 hbo', 'S16_D12 hbo','S16_D14 hbo']
# }


# roi_definitions = {
#     "Left Frontal": [
#         'S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo', 'S2_D3 hbo',
#         'S3_D1 hbo', 'S3_D3 hbo', 'S4_D3 hbo', 'S4_D2 hbo', 'S4_D4 hbo',
#         'S5_D3 hbo', 'S5_D4 hbo', 'S5_D16 hbo'
#     ],
#     "Left Parietal": [
#         'S11_D9 hbo', 'S11_D8 hbo', 'S12_D8 hbo', 'S12_D10 hbo', 'S13_D8 hbo',
#         'S13_D9 hbo', 'S13_D10 hbo', 'S13_D11 hbo', 'S12_D19 hbo', 'S11_D18 hbo', 'S13_D20 hbo'
#     ],
#     "Right Frontal": [
#         'S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo', 'S7_D7 hbo',
#         'S8_D5 hbo', 'S8_D7 hbo', 'S9_D7 hbo', 'S9_D6 hbo', 'S10_D7 hbo', 'S10_D17 hbo'
#     ],
#     "Right Parietal": [
#         'S14_D12 hbo', 'S14_D13 hbo', 'S15_D12 hbo', 'S15_D14 hbo', 'S16_D12 hbo',
#         'S16_D13 hbo', 'S16_D14 hbo', 'S16_D15 hbo', 'S14_D21 hbo', 'S15_D22 hbo', 'S16_D23 hbo'
#     ]
# }

# # Define Regions of Interest (ROI)
# roi_definitions = {
#     "Left RLPFC": [
#         'S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'
#     ],
#     "Left DLPFC": [
#           'S4_D3 hbo', 'S4_D2 hbo', 'S4_D4 hbo','S5_D3 hbo', 'S5_D4 hbo', 'S5_D16 hbo','S2_D3 hbo'
#     ],
#     "Left VLPFC": [
#          'S3_D1 hbo', 'S3_D3 hbo'
#     ],
#     "Left Sup Parietal": [
#          'S12_D8 hbo', 'S12_D10 hbo', 'S13_D10 hbo','S13_D20 hbo', 'S12_D19 hbo'
#     ],
#     "Left Inf Parietal": [
#         'S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo', 'S13_D9 hbo', 'S13_D11 hbo', 'S11_D18 hbo',
#     ],
#     "Right RLPFC": [
#         'S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo', 
#     ],
#     "Right DLPFC": [
#          'S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo', 'S10_D7 hbo', 'S10_D17 hbo'
#     ],
#     "Right VLPFC": [
#          'S9_D7 hbo', 'S9_D6 hbo'
#     ],
#     "Right Sup Parietal": [
#         'S14_D12 hbo', 'S14_D13 hbo', 
#          'S14_D21 hbo', 'S16_D13 hbo'
#     ],
#     "Right Inf Parietal": [
#          'S15_D12 hbo', 'S15_D14 hbo','S16_D12 hbo', 'S16_D14 hbo', 'S16_D15 hbo', 'S15_D22 hbo','S16_D23 hbo' 
#     ]
# }

# # Define Regions of Interest (ROI)
# roi_definitions = {
#     "L-RLPFC": [
#         'S2_D2 hbo'
#     ],
#     "L-DLPFC": [
#           'S4_D3 hbo'
#     ],
#     "L-VLPFC": [
#          'S3_D1 hbo'
#     ],
#     "L-Sup Parietal": [
#           'S12_D10 hbo'
#     ],
#     "L-Inf Parietal": [
#         'S11_D9 hbo'
#     ],
#     "R-RLPFC": [
#         'S7_D5 hbo'
#     ],
#     "R-DLPFC": [
#          'S8_D7 hbo'
#     ],
#     "R-VLPFC": [
#          'S9_D6 hbo'
#     ],
#     "R-Sup Parietal": [
#         'S14_D13 hbo' 
         
#     ],
#     "R-Inf Parietal": [
#          'S15_D14 hbo' 
#     ]
# }

# Optional: Function to normalize channel names (lower-case and stripped)
def normalize_channel(ch):
    return ch.strip().lower()

# Function to look up ROI for a given channel name.
def get_roi(channel_name, roi_definitions):
    # Normalize the channel name for matching
    ch_norm = normalize_channel(channel_name)
    for roi, channels in roi_definitions.items():
        # Normalize each channel name for robust matching
        norm_channels = [normalize_channel(ch) for ch in channels]
        if ch_norm in norm_channels:
            return roi
    return "Unassigned"

# Assume df_hbo is your dataframe that now contains the combined columns "chOrigin" and "chDest".
# (For example, you previously computed these as:
#  df_hbo['chOrigin'] = 'S' + df_hbo['SourceOrigin'].astype(str) + '_D' + df_hbo['DetectorOrigin'].astype(str) + ' ' + df_hbo['TypeOrigin']
#  df_hbo['chDest'] = 'S' + df_hbo['SourceDest'].astype(str) + '_D' + df_hbo['DetectorDest'].astype(str) + ' ' + df_hbo['TypeDest']
#)

# Create new ROI columns by applying the lookup function on the combined channel columns.
df_hbo = df_hbo.copy()  # Create a copy if needed

# Use the new column names for ROI lookup
df_hbo['ROI_X'] = df_hbo['chOrigin'].apply(lambda ch: get_roi(ch, roi_definitions))
df_hbo['ROI_Y'] = df_hbo['chDest'].apply(lambda ch: get_roi(ch, roi_definitions))

# Convert these new ROI columns to categorical variables.
df_hbo['ROI_X'] = df_hbo['ROI_X'].astype('category')
df_hbo['ROI_Y'] = df_hbo['ROI_Y'].astype('category')

## this section is unneccessary keep it commented out 

# # Compute the global r for each subject and add it as a new column
# df_hbo['global_r'] = df_hbo.groupby('subject')['R'].transform('mean')

# # For each row (channel pair), compute the deviation: the difference between the row's R value and the subject's global r
# df_hbo['deviation'] = df_hbo['R'] - df_hbo['global_r']

# #Compute the standard deviation of the deviations for each subject
# df_hbo['std_deviation'] = df_hbo.groupby('subject')['deviation'].transform('std')

# # Calculate the z-score for each channel pair (centered at the subject’s global r)
# df_hbo['z_score'] = df_hbo['deviation'] / df_hbo['std_deviation']
# Now, df_hbo contains your original data plus two new columns 'ROI_X' and 'ROI_Y'
# based on the channel names in 'chOrigin' and 'chDest'.
#df_hbo


In [235]:
# Filtering the data by condition (if you have different conditions as opposed to task vs rest connectivity)
df_cond1 = df_hbo[df_hbo["condition"] == 1]
df_cond2 = df_hbo[df_hbo["condition"] == 2]


In [27]:
## Plot Channel x Channel Matrix with Color Coded Channels based on ROI 

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# # --- Define ROI to channel mappings --- # change to specific ROIs your using
# roi_definitions = {
#     "Left RLPFC": ['S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'],
#     "Left DLPFC": ['S4_D3 hbo', 'S4_D2 hbo', 'S4_D4 hbo', 'S5_D3 hbo', 'S5_D4 hbo', 'S2_D3 hbo'],
#     "Left VLPFC": ['S3_D1 hbo', 'S3_D3 hbo'],
#     "Left Sup Parietal": ['S12_D8 hbo', 'S12_D10 hbo', 'S13_D10 hbo'],
#     "Left Inf Parietal": ['S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo', 'S13_D9 hbo', 'S13_D11 hbo'],
#     "Right RLPFC": ['S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo'],
#     "Right DLPFC": ['S7_D7 hbo', 'S8_D5 hbo', 'S8_D7 hbo', 'S10_D7 hbo'],
#     "Right VLPFC": ['S9_D7 hbo', 'S9_D6 hbo'],
#     "Right Sup Parietal": ['S14_D12 hbo', 'S14_D13 hbo', 'S16_D13 hbo'],
#     "Right Inf Parietal": ['S15_D12 hbo', 'S15_D14 hbo', 'S16_D12 hbo', 'S16_D14 hbo', 'S16_D15 hbo']
# }
# Define ROIs (HbO only)
roi_definitions = {
    "Left RLPFC": ['S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'],
    "Left VLPFC": ['S3_D1 hbo', 'S3_D3 hbo'],
    "Left DLPFC": ['S2_D3 hbo', 'S4_D3 hbo', 'S4_D2 hbo'],
    "Left OFC": ['S1_D15 hbo'],
    "Left STS": ['S5_D4 hbo','S5_D10 hbo','S11_D4 hbo', 'S11_D10 hbo', 'S12_D9 hbo', 'S12_D10 hbo'],
    "Left TPJ": ['S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo','S13_D9 hbo'],
    "Right RLPFC": ['S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo'],
    "Right DLPFC": ['S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo'],
    "Right VLPFC": ['S9_D7 hbo', 'S9_D6 hbo'],
    "Right OFC": ['S6_D15 hbo'],
    "Right STS": ['S10_D13 hbo','S10_D11 hbo','S14_D11 hbo', 'S14_D14 hbo','S15_D11 hbo','S15_D13 hbo'],
    "Right TPJ": ['S15_D12 hbo', 'S15_D14 hbo', 'S16_D12 hbo','S16_D14 hbo']
}

# --- Assign ROI colors ---
roi_colors = sns.color_palette("tab10", n_colors=len(roi_definitions))
roi_color_map = dict(zip(roi_definitions.keys(), roi_colors))

# --- Create channel → ROI map and ordered channel list ---
channel_to_roi = {}
ordered_channels = []
for roi, channels in roi_definitions.items():
    for ch in channels:
        channel_to_roi[ch] = roi
        ordered_channels.append(ch)

# --- Compute mean R values for all channel pairs ---
channel_fc_all = df_hbo.groupby(["chOrigin", "chDest"])["R"].mean().reset_index()

# --- Full grid of all channel pairs ---
full_index = pd.MultiIndex.from_product([ordered_channels, ordered_channels], names=["chOrigin", "chDest"])
full_df = pd.DataFrame(index=full_index).reset_index()
full_matrix = pd.merge(full_df, channel_fc_all, on=["chOrigin", "chDest"], how="left")

# --- Pivot matrix ---
heatmap_data = full_matrix.pivot(index="chOrigin", columns="chDest", values="R")
heatmap_data = heatmap_data.loc[ordered_channels, ordered_channels]

# --- Axis tick colors by ROI ---
x_colors = [roi_color_map[channel_to_roi[ch]] for ch in heatmap_data.columns]
y_colors = [roi_color_map[channel_to_roi[ch]] for ch in heatmap_data.index]

# --- Plot heatmap ---
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    heatmap_data,
    annot=False,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "Mean Functional Connectivity (R)"},
    ax=ax
)

# --- Color axis tick labels ---
for label, color in zip(ax.get_xticklabels(), x_colors):
    label.set_color(color)
for label, color in zip(ax.get_yticklabels(), y_colors):
    label.set_color(color)

# --- Titles and axis formatting ---
title = "Functional Connectivity Matrix"
plt.title(title, fontsize=14)
plt.xlabel("Destination Channel")
plt.ylabel("Origin Channel")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

# --- Create and place legend closer to y-axis ---
handles = [mpatches.Patch(color=color, label=roi) for roi, color in roi_color_map.items()]
fig.legend(
    handles,
    [roi for roi in roi_definitions.keys()],
    title="ROI",
    loc='center left',
    bbox_to_anchor=(0.05, 0.5),
    borderaxespad=0.5,
    frameon=True
)

# --- Save figure ---
save_path = r"C:\Users\pkell\OneDrive\Downloads"  # <-- EDIT THIS PATH
plt.savefig(save_path, dpi=300)
print(f"📁 Figure saved as: {save_path}")
    
plt.show()



📁 Figure saved as: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\36_group_FC_channel.png


In [432]:
## Plot Full Channel × Channel Correlation Matrix (Task vs Rest)

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# --- Compute the mean R values for all channel pairs across conditions ---
channel_fc_all = df_hbo.groupby(["chOrigin", "chDest"])["R"].mean().reset_index()

# --- Get full list of unique channels from both origin and destination ---
all_channels = sorted(set(channel_fc_all["chOrigin"]).union(set(channel_fc_all["chDest"])))

# --- Create a complete channel x channel grid ---
full_index = pd.MultiIndex.from_product([all_channels, all_channels], names=["chOrigin", "chDest"])
full_df = pd.DataFrame(index=full_index).reset_index()

# --- Merge with the actual R values (fill missing with NaN) ---
full_matrix = pd.merge(full_df, channel_fc_all, on=["chOrigin", "chDest"], how="left")

# --- Pivot to create square matrix ---
heatmap_data = full_matrix.pivot(index="chOrigin", columns="chDest", values="R")

# --- Plot the heatmap ---
plt.figure(figsize=(14, 12))
sns.heatmap(
    heatmap_data,
    annot=False,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "Mean Functional Connectivity (R)"}
)

plt.title("Functional Connectivity Matrix (All Channels)")
plt.xlabel("Destination Channel")
plt.ylabel("Origin Channel")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()



In [38]:
## Plot Full ROI × ROI Correlation Matrix (Task vs Rest)

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# --- Convert ROI columns to uppercase to ensure matching ---
df_hbo['ROI_X'] = df_hbo['ROI_X'].str.upper()
df_hbo['ROI_Y'] = df_hbo['ROI_Y'].str.upper()

# --- Desired ROI Order (in uppercase) ---
roi_order = [
    'L-RLPFC', 'L-DLPFC', 'L-VLPFC',
    'R-RLPFC', 'R-DLPFC', 'R-VLPFC',
    'L-INF PARIETAL', 'L-SUP PARIETAL',
    'R-INF PARIETAL', 'R-SUP PARIETAL'
]

# --- Compute mean R for each ROI pair ---
roi_fc_all = df_hbo.groupby(["ROI_X", "ROI_Y"])["R"].mean().reset_index()

# --- Create full grid of ROI combinations to ensure full matrix ---
roi_index = pd.MultiIndex.from_product([roi_order, roi_order], names=["ROI_X", "ROI_Y"])
full_roi_df = pd.DataFrame(index=roi_index).reset_index()

# --- Merge with actual data, keep missing pairs as NaN ---
full_matrix = pd.merge(full_roi_df, roi_fc_all, on=["ROI_X", "ROI_Y"], how="left")

# --- Pivot to square matrix ---
heatmap_data = full_matrix.pivot(index="ROI_X", columns="ROI_Y", values="R")

# --- Set order explicitly to preserve input ROI order ---
heatmap_data = heatmap_data.loc[roi_order, roi_order]

# --- Plot heatmap ---
plt.figure(figsize=(10, 8))
sns.heatmap(
    heatmap_data,
    annot=True,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "Mean Functional Connectivity (R)"}
)

plt.title("Functional Connectivity Matrix (ROI-Level, 73 Participant)")
plt.xlabel("ROI_Y")
plt.ylabel("ROI_X")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# # --- Save figure ---
# save_path = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\36_group_FC_roi.png"  # <-- EDIT THIS PATH
# plt.savefig(save_path, dpi=300)
# print(f"📁 Figure saved as: {save_path}")
    
plt.show()


C:\Users\pkell\AppData\Local\Temp\ipykernel_46076\968711226.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  roi_fc_all = df_hbo.groupby(["ROI_X", "ROI_Y"])["R"].reset_index()


AttributeError: 'SeriesGroupBy' object has no attribute 'reset_index'

In [196]:
## PLot ROI Correlation Matrix for Condition 1 and Conditon 2  

roi_fc = df_hbo.groupby(["ROI_X", "ROI_Y", "condition"])["R"].mean().reset_index()

# Pivot the data for heatmap visualization
import seaborn as sns
import matplotlib.pyplot as plt

for cond in [1, 2]:
    plt.figure(figsize=(8, 6))
    subset = roi_fc[roi_fc["condition"] == cond]
    heatmap_data = subset.pivot(index="ROI_X", columns="ROI_Y", values="R")
    sns.heatmap(heatmap_data, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
    plt.title(f"Functional Connectivity (Condition {cond})")
    plt.xlabel("ROI_Y")
    plt.ylabel("ROI_X")
    plt.show()

C:\Users\pkell\AppData\Local\Temp\ipykernel_46076\3022869140.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  roi_fc = df_hbo.groupby(["ROI_X", "ROI_Y", "condition"])["R"].mean().reset_index()


In [ ]:
## Create Correlation Matrix for All ROI Pairs for Entire Time Series (will result in a bunch of marices )

# List of ROI keys (using your ROI definitions)
rois = list(roi_definitions.keys())

# Loop through each combination of ROI pairs
for roi_x in rois:
    for roi_y in rois:
        # Filter rows for this ROI pair
        subset = df_hbo[(df_hbo['ROI_X'] == roi_x) & (df_hbo['ROI_Y'] == roi_y)]
        if subset.empty:
            print(f"No data for ROI pair: {roi_x} vs {roi_y}")
            continue
        
        # Create a pivot table to form a correlation matrix:
        # Rows: channel names for the first channel ('chOrigin')
        # Columns: channel names for the second channel ('chDest')
        # Values: mean correlation (from the 'R' column)
        corr_matrix = subset.pivot_table(index='chOrigin', 
                                         columns='chDest', 
                                         values='R', 
                                         aggfunc='mean')
        
        # Plot the heatmap for this ROI pair
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", cbar_kws={'label': 'Correlation'}, vmin=0, vmax=.7)
        plt.title(f"Correlation Matrix for ROI: {roi_x} vs {roi_y}")
        plt.xlabel("Channel in ROI_Y")
        plt.ylabel("Channel in ROI_X")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


In [37]:
import seaborn as sns
import matplotlib.pyplot as plt

# Filtering the data by condition
df_cond1 = df_hbo[df_hbo["condition"] == 1]
df_cond2 = df_hbo[df_hbo["condition"] == 2]

sns.histplot(df_cond1["R"], color="blue", label="Condition 1", kde=True, alpha=0.5)
sns.histplot(df_cond2["R"], color="red", label="Condition 2", kde=True, alpha=0.5)
plt.legend()
plt.title("Distribution of Connectivity (R) in Conditions 1 and 2")
plt.xlabel("R")
plt.ylabel("Density")
plt.show()

In [39]:
from scipy.stats import ttest_rel, wilcoxon
# # # Filtering the data by condition
# # df_cond1 = df_hbo1[df_hbo1["condition"] == 1]
# # df_cond2 = df_hbo1[df_hbo1["condition"] == 2]

df_cond1 = df_cond1.dropna()
df_cond2 = df_cond2.dropna()

# Ensure matching channel pairs (both chOrigin and chDest)
assert list(df_cond1["chOrigin"]) == list(df_cond2["chOrigin"]) and \
       list(df_cond1["chDest"]) == list(df_cond2["chDest"]), "Mismatch in channel pairs between conditions"

# Paired t-test (parametric)
t_stat, p_val = ttest_rel(df_cond1["R"], df_cond2["R"])

# Wilcoxon signed-rank test (non-parametric)
wilcox_stat, wilcox_p = wilcoxon(df_cond1["R"], df_cond2["R"])

print(f"Paired t-test: t={t_stat}, p={p_val}")
print(f"Wilcoxon signed-rank test: W={wilcox_stat}, p={wilcox_p}")

mean_r_cond1 = df_hbo[df_hbo["condition"] == 1]["R"].mean()
mean_r_cond2 = df_hbo[df_hbo["condition"] == 2]["R"].mean()

print(f"Mean connectivity (Condition 1): {mean_r_cond1}")
print(f"Mean connectivity (Condition 2): {mean_r_cond2}")

if mean_r_cond1 > mean_r_cond2:
    print("Condition 1 has higher overall connectivity.")
elif mean_r_cond1 < mean_r_cond2:
    print("Condition 2 has higher overall connectivity.")
else:
    print("Connectivity is the same in both conditions.")
    
mean_diff = np.mean(df_cond2["R"] - df_cond1["R"])
std_diff = np.std(df_cond2["R"] - df_cond1["R"], ddof=1)  # Standard deviation of the differences
cohen_d = mean_diff / std_diff
print(f"Cohen's d: {cohen_d}")

Paired t-test: t=15.777049182503756, p=4.870882409829571e-53
Wilcoxon signed-rank test: W=678847.0, p=2.647428541482589e-47
Mean connectivity (Condition 1): 0.35851770022059415
Mean connectivity (Condition 2): 0.35460371167331134
Condition 1 has higher overall connectivity.
Cohen's d: nan


In [237]:
## Correlate vector of t values from 1st order to T vector from second order

from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Ensure data is sorted consistently ---
df_cond1_sorted = df_cond1.sort_values(by=['ROI_X', 'ROI_Y', 'chOrigin', 'chDest']).reset_index(drop=True)
df_cond2_sorted = df_cond2.sort_values(by=['ROI_X', 'ROI_Y', 'chOrigin', 'chDest']).reset_index(drop=True)

# --- Create the r-value vectors ---
r_vec_cond1 = df_cond1_sorted['R'].values
r_vec_cond2 = df_cond2_sorted['R'].values

# --- Double-check that vectors have the same length ---
if len(r_vec_cond1) != len(r_vec_cond2):
    raise ValueError("Condition 1 and Condition 2 vectors do not have the same length! Check preprocessing.")
else:
    print(f"✅ Matched number of channel pairs: {len(r_vec_cond1)}")

# --- Remove pairs where either condition has NaN ---
valid_mask = np.isfinite(r_vec_cond1) & np.isfinite(r_vec_cond2)
r_vec_cond1_clean = r_vec_cond1[valid_mask]
r_vec_cond2_clean = r_vec_cond2[valid_mask]

print(f"✅ Number of valid (non-NaN) pairs: {len(r_vec_cond1_clean)}")

# --- Correlate the cleaned vectors ---
r_across_conditions, p_value_across_conditions = pearsonr(r_vec_cond1_clean, r_vec_cond2_clean)

# --- Print the results ---
print(f"📈 Pearson correlation between Condition 1 and Condition 2 channel-level r values (after removing NaNs):")
print(f"r = {r_across_conditions:.3f}, p = {p_value_across_conditions:.30f}")

# --- Optional: Visualize ---
plt.figure(figsize=(7, 6))
sns.regplot(x=r_vec_cond1_clean, y=r_vec_cond2_clean, scatter_kws={'s':30, 'alpha':0.7})

plt.title('Channel-level FC Correlation: Condition 1 vs Condition 2', fontsize=14)
plt.xlabel('Condition 1 r-values')
plt.ylabel('Condition 2 r-values')
plt.annotate(f"r = {r_across_conditions:.2f}\np = {p_value_across_conditions:.4f}",
             xy=(0.05, 0.95), xycoords='axes fraction',
             ha='left', va='top', fontsize=11,
             bbox=dict(boxstyle="round", facecolor='white', edgecolor='gray', alpha=0.8))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


✅ Matched number of channel pairs: 100
✅ Number of valid (non-NaN) pairs: 90
📈 Pearson correlation between Condition 1 and Condition 2 channel-level r values (after removing NaNs):
r = 0.994, p = 0.000000000000000000000000000000


In [43]:
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Ensure data is sorted consistently ---
df_cond1_sorted = df_cond1.sort_values(by=['ROI_X', 'ROI_Y', 'chOrigin', 'chDest']).reset_index(drop=True)
df_cond2_sorted = df_cond2.sort_values(by=['ROI_X', 'ROI_Y', 'chOrigin', 'chDest']).reset_index(drop=True)

# --- Create the t-value vectors ---
t_vec_cond1 = df_cond1_sorted['t'].values
t_vec_cond2 = df_cond2_sorted['t'].values

# --- Double-check that vectors have the same length ---
if len(t_vec_cond1) != len(t_vec_cond2):
    raise ValueError("Condition 1 and Condition 2 vectors do not have the same length! Check preprocessing.")
else:
    print(f"✅ Matched number of channel pairs: {len(t_vec_cond1)}")

# --- Remove pairs where either condition has NaN ---
valid_mask = np.isfinite(t_vec_cond1) & np.isfinite(t_vec_cond2)
t_vec_cond1_clean = t_vec_cond1[valid_mask]
t_vec_cond2_clean = t_vec_cond2[valid_mask]

print(f"✅ Number of valid (non-NaN) pairs: {len(t_vec_cond1_clean)}")

# --- Correlate the cleaned vectors ---
r_across_conditions, p_value_across_conditions = pearsonr(t_vec_cond1_clean, t_vec_cond2_clean)

# --- Print the results ---
print(f"📈 Pearson correlation between Condition 1 and Condition 2 channel-level t-values (after removing NaNs):")
print(f"r = {r_across_conditions:.3f}, p = {p_value_across_conditions:.4f}")

# --- Optional: Visualize ---
plt.figure(figsize=(7, 6))
sns.regplot(x=t_vec_cond1_clean, y=t_vec_cond2_clean, scatter_kws={'s':30, 'alpha':0.7})

plt.title('Channel-level t-Value Correlation: Condition 1 vs Condition 2', fontsize=14)
plt.xlabel('Condition 1 t-values')
plt.ylabel('Condition 2 t-values')
plt.annotate(f"r = {r_across_conditions:.2f}\np = {p_value_across_conditions:.4f}",
             xy=(0.05, 0.95), xycoords='axes fraction',
             ha='left', va='top', fontsize=11,
             bbox=dict(boxstyle="round", facecolor='white', edgecolor='gray', alpha=0.8))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


✅ Matched number of channel pairs: 2070
✅ Number of valid (non-NaN) pairs: 2070
📈 Pearson correlation between Condition 1 and Condition 2 channel-level t-values (after removing NaNs):
r = 0.995, p = 0.0000


In [ ]:
## subject level correlation between condition 1 and 2 

In [ ]:
# import statsmodels.formula.api as smf

# # Ensure that the columns are in the correct type (categorical for the grouping variables)
# df_hbo['subject'] = df_hbo['subject'].astype('category')
# df_hbo['condition'] = df_hbo['condition'].astype('category')
# df_hbo['ROI_X'] = df_hbo['ROI_X'].astype('category')
# df_hbo['ROI_Y'] = df_hbo['ROI_Y'].astype('category')

# # Define and fit the mixed-effects model:
# # Fixed effects: condition, ROI_X, ROI_Y, and all their interactions.
# # Random effects: random intercept per subject.
# model = smf.mixedlm("correlation ~ C(condition) * C(ROI_X) * C(ROI_Y)",
#                     data=df_hbo,
#                     groups=df_hbo["subject"],
#                     re_formula="1")  # random intercept only

# result = model.fit()
# print(result.summary())


In [ ]:
# import statsmodels.formula.api as smf

# # Make sure that the grouping variables are categorical:
# df_hbo['condition'] = df_hbo['condition'].astype('category')
# df_hbo['ROI_X'] = df_hbo['ROI_X'].astype('category')
# df_hbo['ROI_Y'] = df_hbo['ROI_Y'].astype('category')

# # Fit the OLS model with the interaction between condition, ROI_X, and ROI_Y.
# model = smf.ols("R ~ C(condition) * C(ROI_X) * C(ROI_Y)", data=df_hbo).fit()

# print(model.summary())


In [187]:
import pandas as pd

# Filter for 'hbo' types
df_hbo1 = df1[(df1['TypeOrigin'] == 'hbo') & (df1['TypeDest'] == 'hbo')]

# Create new columns for channel names
df_hbo1['chOrigin'] = 'S' + df_hbo1['SourceOrigin'].astype(str) + '_D' + df_hbo1['DetectorOrigin'].astype(str) + ' ' + df_hbo1['TypeOrigin']
df_hbo1['chDest'] = 'S' + df_hbo1['SourceDest'].astype(str) + '_D' + df_hbo1['DetectorDest'].astype(str) + ' ' + df_hbo1['TypeDest']

# Drop original columns if no longer needed
df_hbo1 = df_hbo1.drop(columns=['SourceOrigin', 'DetectorOrigin', 'TypeOrigin', 'SourceDest', 'DetectorDest', 'TypeDest'])

# Create a column for consistent channel pairs, ensuring mirrored pairs are treated as the same
df_hbo1['pair_key'] = df_hbo1.apply(lambda x: tuple(sorted([x['chOrigin'], x['chDest']])), axis=1)

# Remove mirrored channel pairs within the same subject and condition
df_hbo1 = df_hbo1.drop_duplicates(subset=['subject', 'condition', 'pair_key'])

# Drop the auxiliary pair_key column
df_hbo1 = df_hbo1.drop(columns=['pair_key'])

# Compute the global r for each subject
df_hbo1['global_r'] = df_hbo1.groupby('subject')['R'].transform('mean')

# Compute deviation (R - subject's global R)
df_hbo1['deviation'] = df_hbo1['R'] - df_hbo1['global_r']

# Compute the standard deviation of deviations for each subject
df_hbo1['std_deviation'] = df_hbo1.groupby('subject')['deviation'].transform('std')

# Compute z-score for each channel pair
df_hbo1['z_score'] = df_hbo1['deviation'] / df_hbo1['std_deviation']

# Inspect the modified dataframe
# df_hbo1


C:\Users\pkell\AppData\Local\Temp\ipykernel_17264\2079556593.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hbo1['chOrigin'] = 'S' + df_hbo1['SourceOrigin'].astype(str) + '_D' + df_hbo1['DetectorOrigin'].astype(str) + ' ' + df_hbo1['TypeOrigin']
C:\Users\pkell\AppData\Local\Temp\ipykernel_17264\2079556593.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hbo1['chDest'] = 'S' + df_hbo1['SourceDest'].astype(str) + '_D' + df_hbo1['DetectorDest'].astype(str) + ' ' + df_hbo1['TypeDest']


In [189]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# (Already Provided) -- Preparing df_hbo1
# df_hbo1 = ... (your original code preparing df_hbo1)

# --- Now start the NEW analysis: Correlate Condition 1 vs 2 for each subject ---

# Prepare a list to store correlation results
subject_corrs = []

# Get the list of unique subjects
subjects = df_hbo1['subject'].unique()

for subject in subjects:
    subj_data = df_hbo1[df_hbo1['subject'] == subject]

    # Separate by condition
    df_cond1 = subj_data[subj_data['condition'] == 1].sort_values(by=['chOrigin', 'chDest']).reset_index(drop=True)
    df_cond2 = subj_data[subj_data['condition'] == 2].sort_values(by=['chOrigin', 'chDest']).reset_index(drop=True)

    # Extract R vectors
    r_vec_cond1 = df_cond1['R'].values
    r_vec_cond2 = df_cond2['R'].values

    # Make sure they are the same length
    min_len = min(len(r_vec_cond1), len(r_vec_cond2))
    r_vec_cond1 = r_vec_cond1[:min_len]
    r_vec_cond2 = r_vec_cond2[:min_len]

    # Remove NaNs
    valid_mask = np.isfinite(r_vec_cond1) & np.isfinite(r_vec_cond2)
    r_vec_cond1_clean = r_vec_cond1[valid_mask]
    r_vec_cond2_clean = r_vec_cond2[valid_mask]

    if len(r_vec_cond1_clean) > 2:
        r, p = pearsonr(r_vec_cond1_clean, r_vec_cond2_clean)
        subject_corrs.append({
            'subject': subject,
            'pearson_r': r,
            'p_value': p,
            'n_pairs': len(r_vec_cond1_clean)
        })
    else:
        print(f"⚠️ Not enough valid channel pairs for {subject}. Skipping.")

# --- Convert to DataFrame ---
subject_corrs_df = pd.DataFrame(subject_corrs)
print(subject_corrs_df)

# --- Plot 1: Bar plot of Pearson r for each participant ---
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=subject_corrs_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')

# Add text labels above bars
for i, row in subject_corrs_df.iterrows():
    ax.text(i, row['pearson_r'] + 0.02, f"{row['pearson_r']:.2f}", ha='center', va='bottom', fontsize=9)

plt.title('Session-to-Session r (Condition 1 vs Condition 2) per Participant', fontsize=14)
plt.xlabel('Participant')
plt.ylabel('Pearson r')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1.5)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# --- Plot 2: Histogram of Pearson r values across participants ---
plt.figure(figsize=(8, 5))
sns.histplot(subject_corrs_df['pearson_r'], bins=10, kde=True, color='steelblue', edgecolor='black')

plt.title('Distribution of Participant-Level r Values (Cond1 vs Cond2)', fontsize=14)
plt.xlabel('Pearson r')
plt.ylabel('Number of Participants')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


    subject  pearson_r  p_value  n_pairs
0   allison   0.980466      0.0     1081
1      asha   0.965695      0.0     1081
2   jasmine   0.926477      0.0     1081
3     julia   0.976924      0.0     1081
4    kareem   0.959591      0.0     1081
5      kate   0.974988      0.0     1081
6    lauren   0.978085      0.0     1081
7   natalie   0.966457      0.0     1081
8    nataly   0.970629      0.0     1081
9      nick   0.984517      0.0     1081
10       pk   0.982699      0.0     1081
11       qi   0.978594      0.0     1081
12    ricky   0.967444      0.0     1081
13    ruben   0.973221      0.0     1081
14     theo   0.983905      0.0     1081


C:\Users\pkell\AppData\Local\Temp\ipykernel_17264\3483847839.py:56: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=subject_corrs_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')


In [82]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
import matplotlib.pyplot as plt
import seaborn as sns

# --- Prepare output storage ---
results = []

# --- Loop over each participant ---
for subject in df_hbo1['subject'].unique():
    subj_data = df_hbo1[df_hbo1['subject'] == subject]

    # ✅ Only drop rows with missing R values
    df_cond1 = subj_data[(subj_data['condition'] == 1) & (~subj_data['R'].isna())]
    df_cond2 = subj_data[(subj_data['condition'] == 2) & (~subj_data['R'].isna())]

    if df_cond1.empty or df_cond2.empty:
        print(f"⚠️ Missing one of the conditions for {subject}, skipping.")
        continue

    # Create unique keys for matching channel pairs
    df_cond1['pair'] = list(zip(df_cond1['chOrigin'], df_cond1['chDest']))
    df_cond2['pair'] = list(zip(df_cond2['chOrigin'], df_cond2['chDest']))

    # Find common channel pairs
    common_pairs = set(df_cond1['pair']).intersection(set(df_cond2['pair']))

    if len(common_pairs) < 2:
        print(f"⚠️ Not enough matching channel pairs for {subject}, skipping.")
        continue

    # Subset both dataframes to only matching channel pairs
    df_cond1 = df_cond1[df_cond1['pair'].isin(common_pairs)].sort_values('pair').reset_index(drop=True)
    df_cond2 = df_cond2[df_cond2['pair'].isin(common_pairs)].sort_values('pair').reset_index(drop=True)

    try:
        # Paired t-test
        t_stat, p_val = ttest_rel(df_cond1['R'], df_cond2['R'])

        # Wilcoxon signed-rank test
        wilcox_stat, wilcox_p = wilcoxon(df_cond1['R'], df_cond2['R'])

        # Means
        mean_r_cond1 = df_cond1['R'].mean()
        mean_r_cond2 = df_cond2['R'].mean()

        # Higher connectivity
        if mean_r_cond1 > mean_r_cond2:
            higher_connectivity = "Condition 1"
        elif mean_r_cond2 > mean_r_cond1:
            higher_connectivity = "Condition 2"
        else:
            higher_connectivity = "Equal"

        # Save results
        results.append({
            'subject': subject,
            't_stat': t_stat,
            't_pval': p_val,
            'wilcoxon_stat': wilcox_stat,
            'wilcoxon_pval': wilcox_p,
            'mean_cond1': mean_r_cond1,
            'mean_cond2': mean_r_cond2,
            'higher_connectivity': higher_connectivity,
            'n_channel_pairs': len(common_pairs)
        })

    except Exception as e:
        print(f"❌ Error processing {subject}: {e}")

# --- Compile into DataFrame ---
stats_df = pd.DataFrame(results)
print(stats_df)

# --- Plot if there are any participants ---
if not stats_df.empty:
    plt.figure(figsize=(8,5))
    sns.histplot(stats_df['t_stat'], kde=True, color='skyblue', edgecolor='black')
    plt.title("Distribution of T-Statistics Across Participants", fontsize=14)
    plt.xlabel("t-statistic (Condition 1 vs Condition 2)")
    plt.ylabel("Number of Participants")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # Optional: print a summary
    print("\n✅ Summary of which condition had higher average connectivity per participant:")
    print(stats_df['higher_connectivity'].value_counts())

else:
    print("⚠️ No participants passed the inclusion criteria — no plot was generated.")


C:\Users\pkell\AppData\Local\Temp\ipykernel_17264\854568871.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cond1['pair'] = list(zip(df_cond1['chOrigin'], df_cond1['chDest']))
C:\Users\pkell\AppData\Local\Temp\ipykernel_17264\854568871.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cond2['pair'] = list(zip(df_cond2['chOrigin'], df_cond2['chDest']))
C:\Users\pkell\AppData\Local\Temp\ipykernel_17264\854568871.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

                     subject     t_stat         t_pval  wilcoxon_stat  \
0               allison2 rel  -3.268206   1.116425e-03       238841.0   
1                  pid002rel   2.161724   3.085861e-02       245810.0   
2                  pid003rel -13.700063   1.656103e-39       144314.0   
3                  pid004rel  -9.797059   9.061888e-22       187922.0   
4                  pid008rel  32.541333  1.901715e-162        37448.0   
5                  pid009rel  12.878827   2.065664e-35       153392.0   
6                  pid011rel  17.357848   1.019801e-59       117973.0   
7                  pid013rel  18.074750   5.265408e-64       118208.0   
8                  pid014rel  -3.521501   4.470250e-04       237836.0   
9                  pid015rel   4.088326   4.667177e-05       231476.0   
10                 pid017rel  -0.616541   5.376673e-01       263187.0   
11                 pid018rel -12.829800   3.579769e-35       153748.0   
12                 pid019rel -20.416674   1.305358e

In [89]:
print(stats_df["t_stat"].max())
print(stats_df["t_stat"].min())

32.54133311696963
-20.416673871458407


In [84]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Prepare data for plotting ---
# Melt the stats_df into long format for seaborn
plot_df = stats_df.melt(
    id_vars=['subject'],
    value_vars=['mean_cond1', 'mean_cond2'],
    var_name='Condition',
    value_name='Mean_Connectivity'
)

# Clean up condition labels for nicer plot
plot_df['Condition'] = plot_df['Condition'].replace({
    'mean_cond1': 'Condition 1',
    'mean_cond2': 'Condition 2'
})

# --- Create the Bar Plot ---
plt.figure(figsize=(12, 6))
sns.barplot(
    data=plot_df,
    x='subject',
    y='Mean_Connectivity',
    hue='Condition',
    palette='Set2',
    edgecolor='black'
)

# --- Final Plot Touches ---
plt.title('Mean Connectivity per Participant by Condition', fontsize=16)
plt.xlabel('Participant', fontsize=14)
plt.ylabel('Mean Connectivity (r)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.ylim(plot_df['Mean_Connectivity'].min() - 0.05, plot_df['Mean_Connectivity'].max() + 0.05)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title='Condition')
plt.tight_layout()
plt.show()


In [139]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
file_path = r"C:\Users\pkell\OneDrive\Documents\MATLAB\glm_result_subjectlevel.csv"
df3 = pd.read_csv(file_path)

# Create a unique 'channel' column
df3['channel'] = 'S' + df3['source'].astype(str) + '_D' + df3['detector'].astype(str)

In [145]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
file_path = r"C:\Users\pkell\OneDrive\Documents\MATLAB\glm_result_subjectlevel.csv"
df3 = pd.read_csv(file_path)

# Create a unique 'channel' column
df3['channel'] = 'S' + df3['source'].astype(str) + '_D' + df3['detector'].astype(str)

# Results list
results = []

# Loop through each participant
for subject in df3['subject'].unique():
    subj_data = df3[df3['subject'] == subject]

    # Extract t-values for each condition
    df_cond1 = subj_data[subj_data['cond'] == 'x1'][['channel', 'tstat']].rename(columns={'tstat': 'tstat_c1'})
    df_cond2 = subj_data[subj_data['cond'] == 'x2'][['channel', 'tstat']].rename(columns={'tstat': 'tstat_c2'})

    # Merge on channel to align conditions
    merged = pd.merge(df_cond1, df_cond2, on='channel')

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for {subject}, skipping.")
        continue

    try:
        t_stat, p_val = ttest_rel(merged['tstat_c1'], merged['tstat_c2'])
        wilcox_stat, wilcox_p = wilcoxon(merged['tstat_c1'], merged['tstat_c2'])

        mean_c1 = merged['tstat_c1'].mean()
        mean_c2 = merged['tstat_c2'].mean()

        higher = "Condition 1" if mean_c1 > mean_c2 else "Condition 2" if mean_c2 > mean_c1 else "Equal"

        results.append({
            'subject': subject,
            't_stat': t_stat,
            't_pval': p_val,
            'wilcoxon_stat': wilcox_stat,
            'wilcoxon_pval': wilcox_p,
            'mean_cond1': mean_c1,
            'mean_cond2': mean_c2,
            'higher_connectivity': higher,
            'n_channels': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error processing {subject}: {e}")

# Create DataFrame
stats_df = pd.DataFrame(results)
print(stats_df)

# Plot if results exist
if not stats_df.empty:
    plt.figure(figsize=(8,5))
    sns.histplot(stats_df['t_stat'], kde=True, color='skyblue', edgecolor='black')
    plt.title("Distribution of T-Statistics Across Participants", fontsize=14)
    plt.xlabel("t-statistic (Condition 1 vs Condition 2)")
    plt.ylabel("Number of Participants")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No participants passed the inclusion criteria — no plot was generated.")

import matplotlib.pyplot as plt
import seaborn as sns

# Melt the dataframe for seaborn-friendly plotting
plot_df = stats_df.melt(
    id_vars='subject',
    value_vars=['mean_cond1', 'mean_cond2'],
    var_name='condition',
    value_name='mean_t'
)

# Rename conditions for nicer labels
plot_df['condition'] = plot_df['condition'].map({
    'mean_cond1': 'Condition 1',
    'mean_cond2': 'Condition 2'
})

# Plot
plt.figure(figsize=(12, 6))
sns.barplot(
    data=plot_df,
    x='subject',
    y='mean_t',
    hue='condition',
    palette='Set2',
    edgecolor='black'
)

plt.title("Mean t-value per Participant by Condition", fontsize=14)
plt.xlabel("Participant")
plt.ylabel("Mean t-value")
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend(title='Condition')
plt.show()



                        subject     t_stat        t_pval  wilcoxon_stat  \
0                  allison2 rel  -4.018362  8.551683e-05         5455.0   
1                     pid001rel   8.619791  3.112820e-15         3107.0   
2                     pid002rel   3.092449  2.296032e-03         6114.0   
3                     pid003rel   3.944663  1.137473e-04         5874.0   
4                     pid004rel -10.479388  1.994764e-20         2319.0   
..                          ...        ...           ...            ...   
67  pid071(angelaling)_relmatch  -7.229445  1.275642e-11         3859.0   
68      pid072(ruben) rel match   2.762296  6.325467e-03         5934.0   
69        pid073relmatch(anna1)  -2.174896  3.092074e-02         6746.0   
70      pid074relmatch(Sahana1)  -1.995326  4.749095e-02         7155.0   
71       pid075relmatch(danny1)   4.330733  2.445027e-05         5571.0   

    wilcoxon_pval  mean_cond1  mean_cond2 higher_connectivity  n_channels  
0    2.411988e-05    0.

In [149]:
# --- Mean and range of t-statistics ---
mean_t = stats_df['t_stat'].mean()
min_t = stats_df['t_stat'].min()
max_t = stats_df['t_stat'].max()

print(f"📊 Mean t-statistic: {mean_t:.4f}")
print(f"📉 Minimum t-statistic: {min_t:.4f}")
print(f"📈 Maximum t-statistic: {max_t:.4f}")
print(f"📐 Range of t-statistics: {max_t - min_t:.4f}")

# --- Count of participants by higher connectivity condition ---
connectivity_counts = stats_df['higher_connectivity'].value_counts()

print("\n🔍 Participants with higher connectivity:")
for cond, count in connectivity_counts.items():
    print(f"  {cond}: {count} participants")


📊 Mean t-statistic: -0.3637
📉 Minimum t-statistic: -10.4794
📈 Maximum t-statistic: 15.0743
📐 Range of t-statistics: 25.5536

🔍 Participants with higher connectivity:
  Condition 2: 36 participants
  Condition 1: 36 participants


In [68]:
import os

downloads_path = os.path.join(os.path.expanduser("~"), "Downloads")  # Get Downloads folder path
file_path = os.path.join(downloads_path, "output_filename.csv")  # Define file name

df_hbo1.to_csv(file_path, index=False)  # Save the DataFrame as CSV
print(f"File saved to: {file_path}")


File saved to: C:\Users\pkell\Downloads\output_filename.csv


In [ ]:
import os

downloads_path = os.path.join(os.path.expanduser("~"), "Downloads")  # Get Downloads folder path
file_path = os.path.join(downloads_path, "output_filename1.csv")  # Define file name

df_hbo.to_csv(file_path, index=False)  # Save the DataFrame as CSV
print(f"File saved to: {file_path}")


In [20]:
# Ensure that the columns are in the correct type (categorical for the grouping variables)

df_hbo1["z_score"] = df_hbo1["z_score"].astype("float32")
df_hbo1 = df_hbo1[['subject', 'z_score', 'chOrigin', 'chDest', 'condition']]


# ## ROI MODEL 
# model = smf.mixedlm(
#     "z_score ~ -1 + ROI_X:ROI_Y:condition",  # Fixed effects for each unique ROI pair across conditions
#     data=df_hbo1,
#     groups=df_hbo1["subject"],  # Random intercept per subject
#     re_formula="1"  # Random intercept only
# )

# # CHANNEL LEVEL MODEL
model = smf.mixedlm(
    "z_score ~ -1 + chOrigin:chDest:condition",  # Fixed effects for unique channel pairs across conditions
    data=df_hbo1,
    groups=df_hbo1["subject"],  # Random intercept per subject
    re_formula="1"  # Random intercept only
)



# Fit the model
result = model.fit(method="nm")



#Print the summary
print(result.summary())

MemoryError: Unable to allocate 137. MiB for an array with shape (4233, 4233) and data type float64

In [74]:
# Convert result to a DataFrame
def statsmodels_to_results(model_result):
    """
    Extracts coefficients and statistics from a statsmodels fitted model into a DataFrame.
    """
    summary_df = pd.DataFrame({
        "Coefficient": model_result.fe_params,
        "Std_Error": model_result.bse,
        "T_Value": model_result.tvalues,
        "P_Value": model_result.pvalues
    })
    summary_df["Significant"] = summary_df["P_Value"] < 0.05  # Mark significant results
    return summary_df

# Get the model output as a DataFrame
model_results_df = statsmodels_to_results(result)

# Display results
model_results_df


,Coefficient,Std_Error,T_Value,P_Value,Significant
Group Var,NaN,NaN,NaN,NaN,False
Intercept,0.002013,0.005852,0.343987,0.730856,False
condition[T.2],-0.004026,0.008276,-0.486482,0.626626,False


In [ ]:
# from scipy.stats import ttest_rel, wilcoxon

# # Filtering the data by condition
# df_cond1 = df_hbo[df_hbo["condition"] == 1]
# df_cond2 = df_hbo[df_hbo["condition"] == 2]

# df_cond1 = df_cond1.dropna()
# df_cond2 = df_cond2.dropna()

# # Ensure matching channels
# assert list(df_cond1["chOrigin"]) == list(df_cond2["chOrigin"])  # Ensure channels are aligned

# # Paired t-test (parametric)
# t_stat, p_val = ttest_rel(df_cond1["Z"], df_cond2["Z"])

# # Wilcoxon signed-rank test (non-parametric)
# wilcox_stat, wilcox_p = wilcoxon(df_cond1["Z"], df_cond2["Z"])

# print(f"Paired t-test: t={t_stat}, p={p_val}")
# print(f"Wilcoxon signed-rank test: W={wilcox_stat}, p={wilcox_p}")

# mean_r_cond1 = df_hbo[df_hbo["condition"] == 1]["Z"].mean()
# mean_r_cond2 = df_hbo[df_hbo["condition"] == 2]["Z"].mean()

# print(f"Mean connectivity (Condition 1): {mean_r_cond1}")
# print(f"Mean connectivity (Condition 2): {mean_r_cond2}")

# if mean_r_cond1 > mean_r_cond2:
#     print("Condition 1 has higher overall connectivity.")
# elif mean_r_cond1 < mean_r_cond2:
#     print("Condition 2 has higher overall connectivity.")
# else:
#     print("Connectivity is the same in both conditions.")


In [17]:
from scipy.stats import ttest_rel, wilcoxon
# Filtering the data by condition
df_cond1 = df_hbo1[df_hbo1["condition"] == 1]
df_cond2 = df_hbo1[df_hbo1["condition"] == 2]

# df_cond1 = df_cond1.dropna()
# df_cond2 = df_cond2.dropna()

# Ensure matching channel pairs (both chOrigin and chDest)
assert list(df_cond1["chOrigin"]) == list(df_cond2["chOrigin"]) and \
       list(df_cond1["chDest"]) == list(df_cond2["chDest"]), "Mismatch in channel pairs between conditions"

# Paired t-test (parametric)
t_stat, p_val = ttest_rel(df_cond1["z_score"], df_cond2["z_score"])

# Wilcoxon signed-rank test (non-parametric)
wilcox_stat, wilcox_p = wilcoxon(df_cond1["z_score"], df_cond2["z_score"])

print(f"Paired t-test: t={t_stat}, p={p_val}")
print(f"Wilcoxon signed-rank test: W={wilcox_stat}, p={wilcox_p}")

mean_r_cond1 = df_hbo1[df_hbo1["condition"] == 1]["z_score"].mean()
mean_r_cond2 = df_hbo1[df_hbo1["condition"] == 2]["z_score"].mean()

print(f"Mean connectivity (Condition 1): {mean_r_cond1}")
print(f"Mean connectivity (Condition 2): {mean_r_cond2}")

if mean_r_cond1 > mean_r_cond2:
    print("Condition 1 has higher overall connectivity.")
elif mean_r_cond1 < mean_r_cond2:
    print("Condition 2 has higher overall connectivity.")
else:
    print("Connectivity is the same in both conditions.")
    
mean_diff = np.mean(df_cond2["z_score"] - df_cond1["z_score"])
std_diff = np.std(df_cond2["z_score"] - df_cond1["z_score"], ddof=1)  # Standard deviation of the differences
cohen_d = mean_diff / std_diff
print(f"Cohen's d: {cohen_d}")


Paired t-test: t=3.0212071129459606, p=0.0025188043386921715
Wilcoxon signed-rank test: W=767358421.0, p=0.0003709951648104291
Mean connectivity (Condition 1): 0.002227714922761074
Mean connectivity (Condition 2): -0.0022277149227610434
Condition 1 has higher overall connectivity.
Cohen's d: nan


In [19]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df_cond1["z_score"], color="blue", label="Condition 1", kde=True, alpha=0.5)
sns.histplot(df_cond2["z_score"], color="red", label="Condition 2", kde=True, alpha=0.5)
plt.legend()
plt.title("Distribution of Connectivity (z-scores) in Conditions 1 and 2")
plt.xlabel("z-score")
plt.ylabel("Density")
plt.show()


In [21]:
from scipy.stats import ttest_rel, wilcoxon
# Filtering the data by condition
df_cond1 = df_hbo1[df_hbo1["condition"] == 1]
df_cond2 = df_hbo1[df_hbo1["condition"] == 2]

# df_cond1 = df_cond1.dropna()
# df_cond2 = df_cond2.dropna()

# Ensure matching channel pairs (both chOrigin and chDest)
assert list(df_cond1["chOrigin"]) == list(df_cond2["chOrigin"]) and \
       list(df_cond1["chDest"]) == list(df_cond2["chDest"]), "Mismatch in channel pairs between conditions"

# Paired t-test (parametric)
t_stat, p_val = ttest_rel(df_cond1["R"], df_cond2["R"])

# Wilcoxon signed-rank test (non-parametric)
wilcox_stat, wilcox_p = wilcoxon(df_cond1["R"], df_cond2["R"])

print(f"Paired t-test: t={t_stat}, p={p_val}")
print(f"Wilcoxon signed-rank test: W={wilcox_stat}, p={wilcox_p}")

mean_r_cond1 = df_hbo1[df_hbo1["condition"] == 1]["R"].mean()
mean_r_cond2 = df_hbo1[df_hbo1["condition"] == 2]["R"].mean()

print(f"Mean connectivity (Condition 1): {mean_r_cond1}")
print(f"Mean connectivity (Condition 2): {mean_r_cond2}")

if mean_r_cond1 > mean_r_cond2:
    print("Condition 1 has higher overall connectivity.")
elif mean_r_cond1 < mean_r_cond2:
    print("Condition 2 has higher overall connectivity.")
else:
    print("Connectivity is the same in both conditions.")
    
mean_diff = np.mean(df_cond2["R"] - df_cond1["R"])
std_diff = np.std(df_cond2["R"] - df_cond1["R"], ddof=1)  # Standard deviation of the differences
cohen_d = mean_diff / std_diff
print(f"Cohen's d: {cohen_d}")


Paired t-test: t=4.49943479273366, p=6.8268534310394675e-06
Wilcoxon signed-rank test: W=763928217.0, p=8.226669811065066e-06
Mean connectivity (Condition 1): 0.35723037412869996
Mean connectivity (Condition 2): 0.35577791233726414
Condition 1 has higher overall connectivity.
Cohen's d: nan


In [18]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df_cond1["R"], color="blue", label="Condition 1", kde=True, alpha=0.5)
sns.histplot(df_cond2["R"], color="red", label="Condition 2", kde=True, alpha=0.5)
plt.legend()
plt.title("Distribution of Connectivity (R Values) in Conditions 1 and 2")
plt.xlabel("R")
plt.ylabel("Density")
plt.show()

In [ ]:
from scipy.stats import ttest_rel
import numpy as np

mean_diff = np.mean(df_cond2["z_score"] - df_cond1["z_score"])
std_diff = np.std(df_cond2["z_score"] - df_cond1["z_score"], ddof=1)  # Standard deviation of the differences
cohen_d = mean_diff / std_diff
print(f"Cohen's d: {cohen_d}")


In [ ]:
df_diff = df_hbo1.pivot(index=["chOrigin", "chDest"], columns="condition", values="z_score")
df_diff["diff"] = df_diff[1] - df_diff[2]  # Difference: Condition 1 - Condition 2

# Check overall trend
print(f"Overall mean difference: {df_diff['diff'].mean()}")

# Count how many channels have higher R in Condition 1 vs Condition 2
higher_in_cond1 = (df_diff["diff"] > 0).sum()
higher_in_cond2 = (df_diff["diff"] < 0).sum()

print(f"Number of channels where Condition 1 > Condition 2: {higher_in_cond1}")
print(f"Number of channels where Condition 2 > Condition 1: {higher_in_cond2}")


In [ ]:
from scipy.stats import ttest_rel
import numpy as np

# Compute Cohen's d effect size
def cohens_d(x, y):
    return (np.mean(x) - np.mean(y)) / np.sqrt((np.std(x, ddof=1) ** 2 + np.std(y, ddof=1) ** 2) / 2)

d_value = cohens_d(df_cond1["z_score"], df_cond2["z_score"])
print(f"Cohen's d effect size: {d_value}")


In [20]:
roi_fc = df_hbo.groupby(["ROI_X", "ROI_Y", "condition"])["R"].mean().reset_index()

# Pivot the data for heatmap visualization
import seaborn as sns
import matplotlib.pyplot as plt

for cond in [1, 2]:
    plt.figure(figsize=(8, 6))
    subset = roi_fc[roi_fc["condition"] == cond]
    heatmap_data = subset.pivot(index="ROI_X", columns="ROI_Y", values="R")
    sns.heatmap(heatmap_data, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
    plt.title(f"Functional Connectivity (Condition {cond})")
    plt.xlabel("ROI_Y")
    plt.ylabel("ROI_X")
    plt.show()


C:\Users\pkell\AppData\Local\Temp\ipykernel_26272\3971439068.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  roi_fc = df_hbo.groupby(["ROI_X", "ROI_Y", "condition"])["R"].mean().reset_index()


In [22]:
import seaborn as sns
import matplotlib.pyplot as plt

# Compute the mean R values across all conditions
roi_fc_all = df_hbo.groupby(["ROI_X", "ROI_Y"])["R"].mean().reset_index()

# Pivot the data for heatmap visualization
plt.figure(figsize=(8, 6))
heatmap_data = roi_fc_all.pivot(index="ROI_X", columns="ROI_Y", values="R")

# Plot the heatmap without separating by condition
sns.heatmap(heatmap_data, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Functional Connectivity (All Conditions Combined)")
plt.xlabel("ROI_Y")
plt.ylabel("ROI_X")
plt.show()


C:\Users\pkell\AppData\Local\Temp\ipykernel_26272\3485645283.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  roi_fc_all = df_hbo.groupby(["ROI_X", "ROI_Y"])["R"].mean().reset_index()


In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
df_hb0 = df_hbo.dropna()
# Compute linkage matrix
Z = linkage(df_hb0[["R"]], method='ward')

# Plot dendrogram
plt.figure(figsize=(10, 5))
dendrogram(Z, labels=df_hb0["chOrigin"].values, leaf_rotation=90)
plt.title("Hierarchical Clustering of Connectivity Patterns")
plt.show()


In [ ]:
from sklearn.cluster import KMeans

# Define number of clusters (adjust based on data)
num_clusters = 4
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
df_hb0["Cluster"] = kmeans.fit_predict(df_hb0[["R"]])

# Visualize cluster distributions
sns.boxplot(x="Cluster", y="R", data=df_hb0)
plt.title("Cluster-Based Connectivity Patterns")
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Split data by condition
df_cond1 = df_hb0[df_hb0["condition"] == 1].copy()
df_cond2 = df_hb0[df_hb0["condition"] == 2].copy()

# Define number of clusters (adjust based on data)
num_clusters = 2

# Run K-Means separately for each condition
kmeans_cond1 = KMeans(n_clusters=num_clusters, random_state=42)
kmeans_cond2 = KMeans(n_clusters=num_clusters, random_state=42)

df_cond1["Cluster"] = kmeans_cond1.fit_predict(df_cond1[["R"]])
df_cond2["Cluster"] = kmeans_cond2.fit_predict(df_cond2[["R"]])

# Combine data for comparison
df_hbo_clustered = pd.concat([df_cond1, df_cond2])

# Plot Cluster-Based Connectivity Patterns for Both Conditions
plt.figure(figsize=(10, 6))
sns.boxplot(x="Cluster", y="R", hue="condition", data=df_hbo_clustered)
plt.title("Cluster-Based Connectivity Patterns (Comparison)")
plt.xlabel("Cluster")
plt.ylabel("Functional Connectivity (R)")
plt.legend(title="Condition")
plt.show()


In [ ]:
!pip install hdbscan

In [ ]:
import hdbscan
import seaborn as sns
import matplotlib.pyplot as plt

# How to Interpret the Results
# HDBSCAN assigns a cluster to each channel pair based on their functional connectivity (R values).
# Some channels are labeled as -1 (outliers) if they don't fit into any cluster.
# Compare how cluster memberships shift between Condition 1 and Condition 2:
# If different channels cluster together between conditions, functional networks reconfigure.
# If clusters remain stable, the functional networks are consistent across conditions.
# Outliers (-1 cluster) may indicate weak connectivity or noisy channels.


# Split data by condition
df_cond1 = df_hb0[df_hb0["condition"] == 1].copy()
df_cond2 = df_hb0[df_hb0["condition"] == 2].copy()

# Run HDBSCAN separately for each condition
hdbscan_cond1 = hdbscan.HDBSCAN(min_cluster_size=5, metric="euclidean", cluster_selection_method="eom")
hdbscan_cond2 = hdbscan.HDBSCAN(min_cluster_size=5, metric="euclidean", cluster_selection_method="eom")

df_cond1["Cluster"] = hdbscan_cond1.fit_predict(df_cond1[["R"]])
df_cond2["Cluster"] = hdbscan_cond2.fit_predict(df_cond2[["R"]])

# Combine data for comparison
df_hbo_clustered = pd.concat([df_cond1, df_cond2])

# Plot Cluster-Based Connectivity Patterns for Both Conditions
plt.figure(figsize=(10, 6))
sns.boxplot(x="Cluster", y="R", hue="condition", data=df_hbo_clustered)
plt.title("HDBSCAN Cluster-Based Connectivity Patterns (Comparison)")
plt.xlabel("Cluster")
plt.ylabel("Functional Connectivity (R)")
plt.legend(title="Condition")
plt.show()


In [ ]:
from sklearn.metrics import adjusted_rand_score, silhouette_score
import numpy as np

# Extract cluster labels for both conditions (ignoring outliers labeled as -1)
labels_cond1 = df_cond1["Cluster"].values
labels_cond2 = df_cond2["Cluster"].values

# Remove outliers (-1) before computing metrics
valid_idx = (labels_cond1 != -1) & (labels_cond2 != -1)

# Compute Adjusted Rand Index (ARI)
ari_score = adjusted_rand_score(labels_cond1[valid_idx], labels_cond2[valid_idx])

# Compute Silhouette Score for each condition
silhouette_cond1 = silhouette_score(df_cond1[["R"]], labels_cond1) if len(set(labels_cond1)) > 1 else np.nan
silhouette_cond2 = silhouette_score(df_cond2[["R"]], labels_cond2) if len(set(labels_cond2)) > 1 else np.nan

# Output the results
ari_score, silhouette_cond1, silhouette_cond2


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.metrics import adjusted_rand_score
from hdbscan import HDBSCAN
from collections import Counter
from itertools import product
from matplotlib.sankey import Sankey

# --- Step 1: Compare Cluster Distributions ---
def compare_cluster_distributions(df_hbo_clustered):
    """
    Compares the distribution of clusters between conditions.
    """
    # Count clusters per condition
    cluster_counts = df_hbo_clustered.groupby(["condition", "Cluster"]).size().unstack(fill_value=0)
    
    # Normalize the counts to proportions for better comparison
    cluster_proportions = cluster_counts.div(cluster_counts.sum(axis=1), axis=0)

    # Plot the cluster distributions
    plt.figure(figsize=(10, 6))
    cluster_proportions.T.plot(kind="bar", stacked=True, colormap="viridis", alpha=0.85)
    plt.title("Cluster Distributions by Condition")
    plt.xlabel("Cluster")
    plt.ylabel("Proportion of Channels in Cluster")
    plt.legend(title="Condition")
    plt.xticks(rotation=0)
    plt.show()
    
    return cluster_counts

# --- Step 2: Perform Statistical Tests ---
def test_cluster_significance(cluster_counts):
    """
    Runs a chi-square test to see if cluster distributions significantly differ between conditions.
    """
    # Chi-Square Test
    chi2_stat, p_val, dof, expected = chi2_contingency(cluster_counts)
    print(f"Chi-Square Test:\nStatistic = {chi2_stat:.3f}, p-value = {p_val:.5f}")
    
    if p_val < 0.05:
        print("Significant difference in cluster distributions between conditions!")
    else:
        print("No significant difference in cluster distributions.")

# --- Step 3: Visualize Cluster Shifts (Sankey Diagram) ---
def plot_cluster_transitions(df_hbo_clustered):
    """
    Creates a Sankey diagram to visualize how ROIs shift clusters between conditions.
    """
    # Create a table of cluster transitions
    transition_counts = df_hbo_clustered.pivot_table(index="chOrigin", columns="condition", values="Cluster", aggfunc="first")
    transition_counts = transition_counts.dropna()  # Remove any missing values
    
    # Convert transitions to count pairs
    transition_pairs = transition_counts.value_counts().reset_index()
    transition_pairs.columns = ["Cluster_Cond1", "Cluster_Cond2", "Count"]

    # Plot the Sankey diagram
    sankey = Sankey(flows=transition_pairs["Count"].values,
                    labels=[f"C1:{a} → C2:{b}" for a, b in zip(transition_pairs["Cluster_Cond1"], transition_pairs["Cluster_Cond2"])],
                    orientations=[0] * len(transition_pairs))

    plt.figure(figsize=(8, 5))
    sankey.finish()
    plt.title("Cluster Transitions Between Conditions")
    plt.show()

# --- Load Data (Make sure you have df_hbo from previous clustering steps) ---
# df_hbo should have "condition", "Cluster", and "chOrigin" columns.

# Step 1: Compare Cluster Distributions
cluster_counts = compare_cluster_distributions(df_hbo_clustered)

# Step 2: Run Chi-Square Test
test_cluster_significance(cluster_counts)

# Step 3: Visualize Cluster Transitions
plot_cluster_transitions(df_hbo_clustered)


In [ ]:
def compare_cluster_distributions(df_hbo_clustered):
    """
    Compares the distribution of clusters across conditions and ROIs.
    """
    # Count occurrences of each cluster per condition and ROI
    cluster_counts = df_hbo_clustered.groupby(["condition", "ROI_X", "ROI_Y", "Cluster"]).size().unstack(fill_value=0)

    # Normalize cluster counts by ROI to get proportions
    cluster_proportions = cluster_counts.div(cluster_counts.sum(axis=1), axis=0)

    # Plot cluster proportions for each ROI pairing
    plt.figure(figsize=(14, 7))
    cluster_proportions.T.plot(kind="bar", stacked=True, colormap="viridis", alpha=0.85)
    plt.title("Cluster Distributions by Condition & ROI Pair")
    plt.xlabel("Cluster")
    plt.ylabel("Proportion of Connections in Cluster")
    plt.legend(title="Condition", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.show()

    return cluster_counts

def test_cluster_significance_by_roi(df_hbo_clustered):
    """
    Runs chi-square tests on cluster distributions within each ROI-to-ROI connection
    to determine if specific ROI pairs significantly shift clusters between conditions.
    """
    unique_roi_pairs = df_hbo_clustered.groupby(["ROI_X", "ROI_Y"]).size().index

    significant_rois = []
    
    for roi_x, roi_y in unique_roi_pairs:
        subset = df_hbo_clustered[(df_hbo_clustered["ROI_X"] == roi_x) & (df_hbo_clustered["ROI_Y"] == roi_y)]
        cluster_counts = subset.groupby(["condition", "Cluster"]).size().unstack(fill_value=0)
        
        # Ensure there is more than one cluster present for valid chi-square test
        if cluster_counts.shape[1] > 1:
            chi2_stat, p_val, _, _ = chi2_contingency(cluster_counts)
            print(f"ROI Pair: {roi_x} ↔ {roi_y} | Chi-Square: {chi2_stat:.3f}, p-value: {p_val:.5f}")

            # Track significant ROI pairs
            if p_val < 0.05:
                significant_rois.append((roi_x, roi_y, p_val))

    return significant_rois
    
def plot_roi_cluster_transitions(df_hbo_clustered):
    """
    Creates a Sankey diagram to visualize how ROI pairs shift clusters between conditions.
    """
    # Ensure valid cluster assignments exist
    df_hbo_clustered = df_hbo_clustered.dropna(subset=["Cluster"])  

    # Create a table of cluster transitions
    transition_counts = df_hbo_clustered.pivot_table(index=["ROI_X", "ROI_Y"], columns="condition", values="Cluster", aggfunc=lambda x: x.mode()[0] if not x.mode().empty else np.nan)
    
    # Remove NaN values (i.e., ROI pairs missing in either condition)
    transition_counts = transition_counts.dropna().reset_index()

    # Rename columns for clarity
    transition_counts.columns = ["ROI_X", "ROI_Y", "Cluster_Cond1", "Cluster_Cond2"]

    # Count occurrences of each transition
    transition_counts["Count"] = transition_counts.groupby(["Cluster_Cond1", "Cluster_Cond2"]).transform("count")["ROI_X"]

    # Plot the Sankey diagram
    plt.figure(figsize=(12, 6))
    sankey = Sankey(flows=transition_counts["Count"].values,
                    labels=[f"{roi_x} ↔ {roi_y} | C1:{c1} → C2:{c2}" 
                            for roi_x, roi_y, c1, c2 in zip(transition_counts["ROI_X"], 
                                                            transition_counts["ROI_Y"], 
                                                            transition_counts["Cluster_Cond1"], 
                                                            transition_counts["Cluster_Cond2"])],
                    orientations=[0] * len(transition_counts))
    
    sankey.finish()
    plt.title("ROI-Based Cluster Transitions Between Conditions")
    plt.show()

# Step 1: Compare Cluster Distributions by ROI Pairs
roi_cluster_counts = compare_cluster_distributions(df_hbo_clustered)

# Step 2: Run Statistical Tests for ROI Cluster Shifts
significant_roi_shifts = test_cluster_significance_by_roi(df_hbo_clustered)

# Step 3: Visualize ROI-Specific Cluster Transitions
plot_roi_cluster_transitions(df_hbo_clustered)


In [ ]:
df_hbo_clustered

In [ ]:
import networkx as nx

# Create a graph where nodes are channels and edges are connectivity strengths
G = nx.Graph()

# Add edges based on connectivity strength
threshold = 0.3  # Define a threshold for strong connectivity
for _, row in df_hbo.iterrows():
    if row["R"] > threshold:  # Only include strong connections
        G.add_edge(row["chOrigin"], row["chDest"], weight=row["R"])

# Plot the graph
plt.figure(figsize=(10, 7))
nx.draw_networkx(G, with_labels=True, node_size=100, font_size=8)
plt.title("Functional Connectivity Graph")
plt.show()
